# Final MIA Audit (v2)

Compute the published MIA metric panel (per-shadow ROC AUC, TPR @ {0.01%, 0.1%, 1%} FPR, balanced accuracy) from the existing shadow-score pickles, aggregate as mean ± std across shadows, and compare black-box vs white-box. No retraining.


In [ ]:
import numpy as np
import pickle
from sklearn.metrics import roc_auc_score, roc_curve
from pathlib import Path


In [ ]:
AUDIT_DIR = Path("/home/govindu.srimaan/project/project/experiments/audit")

with open(AUDIT_DIR / "bb_results.pkl", "rb") as f:
    bb = pickle.load(f)
with open(AUDIT_DIR / "wb_results.pkl", "rb") as f:
    wb = pickle.load(f)

bb_scores = np.asarray(bb["shadow_scores"])
bb_members = np.asarray(bb["shadow_members"])
wb_scores = np.asarray(wb["shadow_scores"])
wb_members = np.asarray(wb["shadow_members"])

print("bb shadow_scores:", bb_scores.shape, "bb shadow_members:", bb_members.shape)
print("wb shadow_scores:", wb_scores.shape, "wb shadow_members:", wb_members.shape)


In [ ]:
def compute_panel(scores, members):
    """Compute the published MIA metric panel for one shadow.

    Parameters
    ----------
    scores : 1-D array of shadow scores (higher = more likely member)
    members : 1-D boolean array, True = canary was a member of this shadow

    Returns
    -------
    dict with keys: auc, tpr_at_0.01, tpr_at_0.1, tpr_at_1,
                   bal_acc, n_members, n_nonmembers
    """
    scores = np.asarray(scores, dtype=float)
    members = np.asarray(members, dtype=bool)
    y = members.astype(int)
    n_members = int(members.sum())
    n_nonmembers = int((~members).sum())

    # Degenerate edge case: only one class present -> cannot compute ROC.
    if n_members == 0 or n_nonmembers == 0:
        nan = float("nan")
        return {
            "auc": nan,
            "tpr_at_0.01": nan,
            "tpr_at_0.1": nan,
            "tpr_at_1": nan,
            "bal_acc": nan,
            "n_members": n_members,
            "n_nonmembers": n_nonmembers,
        }

    auc = float(roc_auc_score(y, scores))

    fpr, tpr, _ = roc_curve(y, scores)
    # roc_curve prepends (0,0) so it always has >= 2 points, but guard anyway.
    if len(fpr) < 2:
        tpr_0001 = float("nan")
        tpr_001 = float("nan")
        tpr_01 = float("nan")
    else:
        # fpr is non-decreasing along the ROC curve; interpolate TPR at
        # the three fixed low-FPR operating points.
        tpr_0001 = float(np.interp(0.0001, fpr, tpr))
        tpr_001 = float(np.interp(0.001, fpr, tpr))
        tpr_01 = float(np.interp(0.01, fpr, tpr))

    # Balanced accuracy: (TPR + TNR) / 2 at the optimal Youden-J point.
    youden = tpr - fpr
    i = int(np.argmax(youden))
    tpr_opt = float(tpr[i])
    tnr_opt = float(1.0 - fpr[i])
    bal_acc = (tpr_opt + tnr_opt) / 2.0

    return {
        "auc": auc,
        "tpr_at_0.01": tpr_0001,
        "tpr_at_0.1": tpr_001,
        "tpr_at_1": tpr_01,
        "bal_acc": bal_acc,
        "n_members": n_members,
        "n_nonmembers": n_nonmembers,
    }


In [ ]:
METRIC_KEYS = ["auc", "tpr_at_0.01", "tpr_at_0.1", "tpr_at_1", "bal_acc"]

def audit_all(scores, members, label):
    per_shadow = [compute_panel(scores[i], members[i]) for i in range(scores.shape[0])]
    print(f"\n===== {label} ({scores.shape[0]} shadows x {scores.shape[1]} canaries) =====")
    header = f"{'shadow':>6} " + " ".join(f"{k:>12}" for k in METRIC_KEYS) + f" {'n_mem':>5} {'n_non':>5}"
    print(header)
    for i, p in enumerate(per_shadow):
        row = " ".join(f"{p[k]:>12.4f}" for k in METRIC_KEYS)
        print(f"{i:>6} {row} {p['n_members']:>5} {p['n_nonmembers']:>5}")
    agg = {}
    for k in METRIC_KEYS:
        vals = np.array([p[k] for p in per_shadow], dtype=float)
        agg[k] = dict(mean=float(np.nanmean(vals)), std=float(np.nanstd(vals)))
    print(f"\n{label} aggregate (mean +/- std across shadows):")
    for k in METRIC_KEYS:
        print(f"  {k:>12}: {agg[k]['mean']:.4f} +/- {agg[k]['std']:.4f}")
    return per_shadow, agg

bb_per_shadow, bb_agg = audit_all(bb_scores, bb_members, "BLACK-BOX")
wb_per_shadow, wb_agg = audit_all(wb_scores, wb_members, "WHITE-BOX")


## Interpretation

This notebook computes the *published* MIA metric panel — per-shadow ROC AUC, TPR @ {0.01%, 0.1%, 1%} FPR, and balanced accuracy — directly from the pre-computed `shadow_scores` / `shadow_members` matrices in `bb_results.pkl` and `wb_results.pkl`, following the conventions documented in `audit_methodology.md` (GLiRA §14 / IMIA §35): AUC alone is not informative, and the low-FPR operating points are the primary risk indicator.

**How to read the table above:**

- **AUC > 0.5 for both bb and wb?** Inspect the `auc` column. AUC strictly above 0.5 for both attack types confirms the shadow scores carry *some* membership signal; AUC ≈ 0.5 would be the empirical signature of a well-defended model (IMIA Table 16: balanced accuracy collapses toward 50% as the DP noise multiplier grows).
- **ERM vs DRO / black-box vs white-box gap.** The pickles do **not** carry an ERM-vs-DRO split (each pickle is a single DP-SGD target's shadow-score matrix), so an ERM↔DRO AUC gap is *not* visible from these artifacts. The comparison this notebook *can* make is **black-box vs white-box** on the same target: a meaningful white-box advantage (higher AUC / higher TPR@low-FPR) would indicate that the shadow scores exploit internal state the black-box attack cannot see; near-parity would suggest the black-box likelihood-ratio score already captures most of the available signal.
- **TPR @ 0.1% FPR — practical attack risk.** This is the operating point both papers treat as primary (GLiRA Table I; IMIA Table 16). Per IMIA's DP-SGD reference values, TPR @ 0.1% FPR ≈ 3.4% for an undefended ResNet on CIFAR-10, dropping to ≈ 0.1–0.3% at σ ≥ 0.5. If our TPR @ 0.1% FPR is in the single-digit percent range, the attack is *practically effective* at the low-false-positive regime a real adversary would operate in; if it is at or below ~0.1%, the DP defense is doing its job at that operating point.
- **Balanced accuracy.** Reported as a threshold-free secondary metric (IMIA §35). Values near 50% corroborate "near-chance" AUC; values well above 50% flag a genuine, exploitable membership signal.

**Caveats (per `audit_methodology.md` "Gaps" section):** the 16 black-box and 7 white-box shadows here were not trained under the published shadow-model protocol (same-distribution subsets, architecture ≥ target, KD from the target), so the absolute numbers are a *lower-bound audit* of the existing artifacts, not a re-derivation of the published attack. No retraining is performed in this notebook by design.


In [ ]:
import json

summary = {
    "n_bb_shadows": int(bb_scores.shape[0]),
    "n_wb_shadows": int(wb_scores.shape[0]),
    "n_canaries": int(bb_scores.shape[1]),
    "bb_panel_mean": {k: bb_agg[k]["mean"] for k in METRIC_KEYS},
    "bb_panel_std": {k: bb_agg[k]["std"] for k in METRIC_KEYS},
    "wb_panel_mean": {k: wb_agg[k]["mean"] for k in METRIC_KEYS},
    "wb_panel_std": {k: wb_agg[k]["std"] for k in METRIC_KEYS},
}

out_path = Path("/home/govindu.srimaan/project/final_audit_v2_summary.json")
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print("Wrote", out_path)
print(json.dumps(summary, indent=2))
